# CARF: A Cybernetic Agent Reliability Framework for APEX-Agents

This notebook is a working proof of concept for CARF, submitted alongside a proposal to the Mercor Research Fellowship extending Mercor's APEX-Agents benchmark.

**Background.** APEX-Agents' judge grades an agent's final output, but does not examine the trajectory that produced it. Two agents can reach the same score by very different routes: one steadily catching and fixing its own mistakes, the other drifting into the right answer without ever checking its work.

**What this notebook demonstrates.** On one small task, it classifies the operational units and environmental elements the task requires, directly from the prompt. It then runs the task step by step and tags each step with the unit it fulfils and the element(s) it draws on. A judge examines the full tagged trajectory and traces whether each step correctly used the element it depended on, catching a drift between steps that a final-answer-only judge would miss. The judge's finding is then turned into feedback and sent back to the agent, which retries the affected step.

**Repository:** [github.com/higginsonlabs/CARF](https://github.com/higginsonlabs/CARF)

---

### Requirements to run this notebook
A free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey), no card required. Add it to Colab's Secrets panel (the key icon in the left sidebar), name it `GEMINI_API_KEY`, and enable notebook access for it. The key stays private to the person running the notebook. It is never written into this notebook or stored in the repository.

## 1. Setup

In [ ]:
!pip install -q google-generativeai

In [ ]:
import json
import re
from google.colab import userdata
import google.generativeai as genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-3.5-flash-lite'  # falls back automatically below if unavailable on your account

def _call_json(prompt: str) -> dict:
    model = genai.GenerativeModel(MODEL_NAME)
    response = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(response_mime_type='application/json'),
    )
    return json.loads(response.text)

print('Setup complete. Model:', MODEL_NAME)

## 2. The task

The task is a small, hand-checkable finance problem, split into four explicit steps so the run produces a real trajectory for the judge to examine. In a full system, an agent would infer this step breakdown itself rather than being given it; the breakdown is made explicit here to keep the prototype simple. The reasoning behind this task is set out in `CARF_DEMO_PLAN.md` in the repository.

In [ ]:
TASK_PROMPT = '''A merger has a bid premium of 20% and cash consideration of 15%.

**Step A:** State these deal terms clearly, on their own.
**Step B:** Using the terms from Step A, calculate the implied offer price on a $40 share price.
**Step C:** Using the terms from Step A, calculate the cash portion of a $10,000 investment.
**Step D:** Summarise both results.
'''
print(TASK_PROMPT)

## 3. Classify operational units and environmental elements

Both are inferred automatically from the task prompt, before any step is run. This mirrors the operational units and environmental elements shown in the CARF architecture diagram included in the proposal.

In [ ]:
def classify_operational_units(prompt: str) -> list:
    call_prompt = f'''You are analysing a task prompt to identify the distinct
operational units it requires: the discrete pieces of work an agent would
need to perform to complete it, in the order they would naturally occur.

Task prompt:
{prompt}

Label each unit "1a", "1b", "1c", ... in order. There is no fixed number of
units. Infer however many the task actually requires. Give each a short
description (a few words).

Respond in JSON with exactly this shape:
{{"units": [{{"id": "1a", "description": "..."}}, ...]}}
'''
    return _call_json(call_prompt)['units']

def classify_environmental_elements(prompt: str) -> list:
    call_prompt = f'''You are analysing a task prompt to identify the specific
facts and data values it depends on: figures, rates, or terms explicitly
given in the prompt that any correct solution must use consistently.

Task prompt:
{prompt}

Label each element "1E", "2E", "3E", ... in the order they appear. There is
no fixed number. Infer however many the task actually gives. Give each a
short description and its stated value.

Respond in JSON with exactly this shape:
{{"elements": [{{"id": "1E", "description": "...", "value": "..."}}, ...]}}
'''
    return _call_json(call_prompt)['elements']

units = classify_operational_units(TASK_PROMPT)
elements = classify_environmental_elements(TASK_PROMPT)

print('Operational units:'); print(json.dumps(units, indent=2))
print('\nEnvironmental elements:'); print(json.dumps(elements, indent=2))

## 4. Run the trajectory, tagged

Steps A to D run as separate calls, each seeing only the steps before it. Each step's output states which operational unit it fulfils and which environmental element(s) it draws on. This tagging is what allows the judge to trace dependencies between steps rather than only checking the final answer.

Setting `INDUCE_DRIFT = True` below reproduces the demo's engineered failure: Step C's instruction is deliberately altered to use an incorrect rate, producing a real, attributable drift for the judge to catch. This is a manufactured failure for demonstration purposes, not an error the model produced on its own. Set it to `False` to run the task cleanly instead.

In [ ]:
INDUCE_DRIFT = True  #@param {type:'boolean'}

DRIFT_STEP_LABEL = 'C'
DRIFT_INJECTION = (
    ' (Use a rate of 22% for this calculation, not the cash consideration '
    'percentage stated earlier.)'
)

def _format_units(units):
    return '\n'.join(f"{u['id']} - {u['description']}" for u in units)

def _format_elements(elements):
    return '\n'.join(f"{e['id']} - {e['description']} ({e.get('value', '')})" for e in elements)

def _parse_steps(task_prompt: str):
    matches = list(re.finditer(r'\*\*Step ([A-Z]):\*\*\s*(.+)', task_prompt))
    premise = task_prompt[: matches[0].start()].strip()
    steps = [{'label': m.group(1), 'instruction': m.group(2).strip()} for m in matches]
    return premise, steps

def _run_step(premise, units, elements, prior_steps, step_label, step_instruction, extra_feedback=None):
    prior_text = '\n'.join(
        f"Step {s['step']} (unit {s['unit']}): {s['output']}" for s in prior_steps
    ) or '(none, this is the first step)'
    feedback_block = f"\n\nFeedback from a previous attempt at this step, address it before answering:\n{extra_feedback}" if extra_feedback else ''
    call_prompt = f'''You are completing a finance task step by step. Only use
the information given below. Do not introduce new facts or figures.

Task premise:
{premise}

Classified operational units for this task:
{_format_units(units)}

Classified environmental elements (facts) for this task:
{_format_elements(elements)}

Steps completed so far, in order:
{prior_text}

Now complete this step:
Step {step_label}: {step_instruction}{feedback_block}

Respond in JSON with exactly this shape:
{{"output": "<your answer to this step, including any calculation>",
  "unit": "<the id from the operational units list that this step fulfils>",
  "elements_used": ["<ids from the environmental elements list you drew on>"]}}
'''
    result = _call_json(call_prompt)
    return {'step': step_label, 'output': result['output'], 'unit': result['unit'], 'elements_used': result['elements_used']}

def run_trajectory(task_prompt, units, elements):
    premise, steps = _parse_steps(task_prompt)
    trajectory = []
    for step in steps:
        instruction = step['instruction']
        if INDUCE_DRIFT and step['label'] == DRIFT_STEP_LABEL:
            instruction += DRIFT_INJECTION
        record = _run_step(premise, units, elements, trajectory, step['label'], instruction)
        trajectory.append(record)
    return trajectory

if INDUCE_DRIFT:
    print('!! ENGINEERED DRIFT MODE: Step C has been deliberately corrupted for this demo run !!\n')

trajectory = run_trajectory(TASK_PROMPT, units, elements)
print(json.dumps(trajectory, indent=2))

## Reference: a verified run

For reference, this is an actual, verified output from this code (Gemini 3.5 Flash Lite), with `INDUCE_DRIFT = True`, saved as `runs/run_002_engineered_drift.json` in the repository.

Step A stated the terms: bid premium 20% (`1E`), cash consideration 15% (`2E`). Step C, deliberately altered for this run, computed `$10,000 * 22% = $2,200`, disregarding `2E`'s true 15% value. The judge returned `judge_grade: "fail"`, `score: 0.75`, with the rationale: "Step C drifted from the established environmental element 2E. While element 2E states cash consideration is 15%, Step C calculates the cash portion using 22% ($10,000 * 0.22 = $2,200), conflicting with the stated parameters." The generated feedback read: "Revise Step C to calculate the cash portion using the 15% rate specified in environmental element 2E instead of 22%." Attempt 2 then corrected to `$10,000 * 15% = $1,500`, matching `2E` exactly.

Running the cells below reproduces this. Results may vary slightly between runs, since the model is not deterministic.

## 5. Judge the trajectory

A single judge call is given the whole tagged trajectory, not just the final step. It is asked to identify failures, assess dependencies (whether each step correctly used the element it depended on), and generate a structured reliability signal. The output is shaped like Mercor's Archipelago `GradeResult` format, since a `type: "trajectory"` verifier is listed there as not yet implemented.

In [ ]:
def judge_trajectory(trajectory, units, elements):
    trajectory_text = '\n'.join(
        f"Step {s['step']} - unit {s['unit']}, elements used {s['elements_used']}: {s['output']}"
        for s in trajectory
    )
    call_prompt = f'''You are the CARF Coordination Judge. You are given a
task's classified operational units, environmental elements, and the full
tagged trajectory of an agent completing the task step by step.

Operational units:
{_format_units(units)}

Environmental elements:
{_format_elements(elements)}

Tagged trajectory:
{trajectory_text}

Your job:
1. Identify failures: did any step's output conflict with an earlier step or
   with the environmental elements' stated values?
2. Assess dependencies: for each step, trace whether it correctly used the
   environmental element(s) it was tagged as depending on, or whether a value
   silently drifted from what was established earlier in the trajectory.
3. Generate a reliability signal.

The valid step labels are exactly: {', '.join(s['step'] for s in trajectory)}.

Respond in JSON with exactly this shape:
{{"judge_grade": "pass" or "fail",
  "score": <float between 0.0 and 1.0>,
  "grade_rationale": "<specific, citing step labels and element ids, explaining any drift found>",
  "affected_step": "<the single letter of the step most responsible, e.g. \\"C\\", or null if judge_grade is pass>"}}
'''
    return _call_json(call_prompt)

judge_result = judge_trajectory(trajectory, units, elements)
print(json.dumps(judge_result, indent=2))

## 6. Close the loop: feedback and re-run

If the judge found a failure, its rationale is turned into one short corrective sentence, sent back to the agent, and the affected step is re-run with that feedback included. This is the interventional half of the CARF feedback loop, alongside the evaluative use of the same signal for grading.

In [ ]:
def close_loop(task_prompt, trajectory, judge_result, units, elements):
    known_labels = [s['step'] for s in trajectory]
    raw_affected = judge_result.get('affected_step') or ''
    affected_label = next((t for t in re.findall(r'[A-Za-z]+', raw_affected) if t in known_labels), None)
    if judge_result.get('judge_grade') == 'pass' or not affected_label:
        return {'needed': False, 'reason': 'judge found no failure to correct'}

    feedback_prompt = f'''Based on this judge finding, write ONE short,
specific corrective sentence to send back to the agent before it retries the
affected step.

Judge finding: {judge_result['grade_rationale']}

Respond in JSON with exactly this shape: {{"feedback": "<one sentence>"}}
'''
    feedback = _call_json(feedback_prompt)['feedback']

    attempt_1 = next(s for s in trajectory if s['step'] == affected_label)
    step_index = next(i for i, s in enumerate(trajectory) if s['step'] == affected_label)
    prior_steps = trajectory[:step_index]

    premise, steps = _parse_steps(task_prompt)
    step_instruction = next(s['instruction'] for s in steps if s['label'] == affected_label)

    attempt_2 = _run_step(premise, units, elements, prior_steps, affected_label, step_instruction, extra_feedback=feedback)

    return {'needed': True, 'affected_step': affected_label, 'feedback': feedback, 'attempt_1': attempt_1, 'attempt_2': attempt_2}

loop_result = close_loop(TASK_PROMPT, trajectory, judge_result, units, elements)
print(json.dumps(loop_result, indent=2))

## 7. Summary

Running the cell below prints a before and after summary of this run.

In [ ]:
print('TASK:', TASK_PROMPT.split(chr(10))[0])
print()
print('JUDGE VERDICT:', judge_result['judge_grade'], '| score:', judge_result['score'])
print('RATIONALE:', judge_result['grade_rationale'])
print()
if loop_result.get('needed'):
    print(f"AFFECTED STEP: {loop_result['affected_step']}")
    print(f"  Attempt 1: {loop_result['attempt_1']['output']}")
    print(f"  Feedback:  {loop_result['feedback']}")
    print(f"  Attempt 2: {loop_result['attempt_2']['output']}")
else:
    print('No correction needed. Judge found the trajectory consistent.')